# 模型使用：从 ICL 到 CoT 与可验证推理

> **本章定位**：在模型参数保持冻结的条件下，使用 In-context Learning（ICL，上下文学习）、Chain-of-Thought（CoT，思维链）和 Self-Consistency（自洽性采样）组织任务求解，并通过统一契约评测最终答案、可验证步骤、延迟和 Token 成本。

ICL 通过运行时示例定义任务，CoT 使用中间步骤组织复杂求解；二者均不构成事实正确性的保证。原始推理轨迹、可验证证据和面向用户的解释需要保持边界。

```mermaid
flowchart LR
    Q["Task Input"] --> S["Example Selection"]
    E["Versioned Example Bank"] --> S
    S --> P["Prompt / Chat Template"]
    P --> G["Generate Candidates"]
    G --> X["Extract Final Answer"]
    X --> V["Verifier / Metric"]
    V --> R["Answer + Evidence + Trace ID"]
```

本章聚焦提示与测试时策略；CoT 数据、训练、Verifier、RLVR 与蒸馏见 `A30_reasoning_model.ipynb`。


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 应用与智能体工程：上下文与推理策略 |
| 本章定位 | 学习不更新参数的模型使用方法，并用统一输出契约评测提示策略。 |
| 先修知识 | 掌握 `31` 的 Chat Template 与生成参数、`50` 的评测集隔离，并阅读 `70` 的输入信任边界；服务发布结合 `60`、`70`。 |
| 预计时间 | 60～90 分钟 |
| 运行资源 | 首次运行下载小型指令模型；CPU 可运行但速度较慢。 |
| 输入 | 任务问题、版本化示例库和 Token Budget。 |
| 交付物 | Zero/Few-shot、CoT、Self-Consistency 结果与可比较指标。 |

### 1.1．学习目标

完成本章后，读者能够构建 Zero-shot、Few-shot ICL、Zero-shot CoT 与 Few-shot CoT 提示，按预算选择示例，使用 Self-Consistency 聚合最终答案，并在统一评测集上比较质量、解析失败、延迟和成本。


### 1.2．环境与依赖

Prompt Builder、解析器和投票逻辑使用 Python 标准库；模型单元使用 PyTorch 与 Transformers，并通过模型自带 Chat Template 组装消息。模型按 ID 直接加载，设备选择兼容 Apple Silicon、CUDA 与 CPU。


## 2．直觉与输入输出契约

### 2.1．ICL、CoT 与运行时上下文

![架构图：不同提示策略构造运行时上下文并输入同一个冻结模型](assets/figures/91_prompt_reasoning/prompting-context.svg)

[TikZ 源文件](assets/figures/91_prompt_reasoning/prompting-context.tex)

ICL 示例是运行时配置资产，应记录版本、来源、适用任务、预期输出和安全标签。示例库不得混入不可信用户文本或真实秘密。本章按 Jaccard 重合度排序并固定选取前 2 条，用于说明示例选择接口；生产环境应在 Token 预算内联合考虑相关性、覆盖与多样性，并设置低相关回退。示例数量需同时满足相关性、覆盖、多样性与 Token 预算；在测试集上反复调整示例会造成评测泄漏。


In [ ]:
import re
from collections import Counter
from dataclasses import dataclass


@dataclass(frozen=True)
class MyReasoningExample:
    """保存可检索的推理示例、简洁步骤、答案与标签。"""
    example_id: str
    question: str
    concise_steps: tuple[str, ...]
    final_answer: str
    tags: frozenset[str]


EXAMPLE_BANK = [
    MyReasoningExample(
        "arith-001",
        "仓库有 12 箱零件，每箱 8 个，取走 20 个后还剩多少？",
        ("总数是 12 × 8 = 96。", "剩余是 96 - 20 = 76。"),
        "76",
        frozenset({"算术", "乘法", "减法"}),
    ),
    MyReasoningExample(
        "arith-002",
        "服务每秒处理 30 个请求，运行 4 秒共处理多少？",
        ("总量等于速率乘时间。", "30 × 4 = 120。"),
        "120",
        frozenset({"算术", "乘法", "吞吐"}),
    ),
    MyReasoningExample(
        "capacity-001",
        "每张卡可用显存 40 GiB，模型需要 95 GiB，至少几张卡？",
        ("卡数向上取整。", "ceil(95 / 40) = 3。"),
        "3",
        frozenset({"算术", "容量", "显存"}),
    ),
]


def my_terms(text: str) -> set[str]:
    """抽取英文词、汉字和数字，形成轻量检索词集合。"""
    return set(re.findall(r"[A-Za-z]+|[\u4e00-\u9fff]|\d+", text.casefold()))


def my_example_score(question: str, example: MyReasoningExample) -> float:
    """以 Jaccard 重叠度衡量问题与示例标签文本的相关性。"""
    query_terms = my_terms(question)
    example_terms = my_terms(example.question + " " + " ".join(example.tags))
    return len(query_terms & example_terms) / max(len(query_terms | example_terms), 1)


# 默认选取 2 个 ICL 示例以验证格式并控制输入占用；生产按相关性、覆盖与总 Token 预算重定。
def my_select_examples(question: str, count: int = 2) -> list[MyReasoningExample]:
    """按相关性和稳定 ID 排序选择指定数量的 ICL 示例。"""
    ranked = sorted(EXAMPLE_BANK, key=lambda item: (-my_example_score(question, item), item.example_id))
    return ranked[:count]


QUESTION = "一台服务每秒生成 25 个 Token，运行 6 秒生成多少 Token？"
selected_examples = my_select_examples(QUESTION)
[(item.example_id, my_example_score(QUESTION, item)) for item in selected_examples]


<!-- theory-math-contract:v1 -->
### 2.2．核心机制的语言与数学表达

Self-Consistency 通过多次随机解码得到候选最终答案，并用多数票近似边缘化不同推理路径：

$$
\hat y=\arg\max_{a\in\mathcal A}\sum_{j=1}^{K}\mathbf 1\!\left(\operatorname{answer}(z_j)=a\right),
\qquad z_j\sim p_\theta(z\mid x;\tau)
$$

其中，$x$ 是提示，$z_j$ 是第 $j$ 条生成轨迹，$K$ 是采样次数，$\tau$ 是温度，$\mathcal A$ 是解析后的候选答案集合。`extract_final_answer` 对应答案解析，计数器对应投票聚合。增大 $K$ 会线性增加 Token 成本，且只有当轨迹具有一定独立性、解析稳定并且错误不高度相关时才可能改善结果。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．四种提示策略与统一输出契约

为了可比较，四种策略都要求最后一行使用 `最终答案：...`。CoT 示例只提供解决任务所需的简洁、可验证步骤；生产系统不应要求模型暴露不可控的冗长内部推理，也不能把生成的推理文字当作真实因果记录。


In [ ]:
def my_format_example(example: MyReasoningExample, include_steps: bool) -> str:
    """把推理示例渲染为含可选步骤的统一文本格式。"""
    if include_steps:
        steps = "\n".join(f"步骤 {index}：{step}" for index, step in enumerate(example.concise_steps, 1))
        return f"问题：{example.question}\n{steps}\n最终答案：{example.final_answer}"
    return f"问题：{example.question}\n最终答案：{example.final_answer}"


def my_build_user_prompt(
    question: str,
    examples: list[MyReasoningExample],
    include_steps: bool,
) -> str:
    """组合任务约束、可选示例和待解问题形成用户 Prompt。"""
    parts = ["请解决问题。最后一行必须使用“最终答案：<答案>”。"]
    if include_steps:
        parts.append("先给出简洁、可验证的计算步骤，再给出最终答案。")
    if examples:
        parts.append("参考示例：")
        parts.extend(my_format_example(example, include_steps) for example in examples)
    parts.append(f"待解决问题：\n{question}")
    return "\n\n".join(parts)


strategies = {
    "zero_shot": my_build_user_prompt(QUESTION, [], False),
    "few_shot_icl": my_build_user_prompt(QUESTION, selected_examples, False),
    "zero_shot_cot": my_build_user_prompt(QUESTION, [], True),
    "few_shot_cot": my_build_user_prompt(QUESTION, selected_examples, True),
}
strategies["few_shot_cot"]


### 3.2．指令模型接口

首次运行下载小型模型。Chat Template 属于模型协议，角色分隔符由模型自带模板生成；Apple Silicon、CUDA 和 CPU 使用统一设备选择。


In [ ]:
import random

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, LogitsProcessor, LogitsProcessorList

# 42 仅固定采样伪随机序列；正式结果使用预注册多 Seed，禁止选择性报告。
SEED = 42
DEVICE = torch.accelerator.current_accelerator(check_available=True) or torch.device("cpu")
# 135M 参数模型用于受限资源下的接口验证；更换模型后须重验示例、解析器和生成预算。
MODEL_ID = "HuggingFaceTB/SmolLM2-135M-Instruct"
# 96 Token 覆盖简短步骤与答案；调低会增加截断，调高会增加延迟与无关生成风险。
# 应在应用 Chat Template 后校验 System、示例、问题与输出预留不超过模型上下文。
MAX_NEW_TOKENS = 96

random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
elif DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
).to(DEVICE)
model.eval()


class MyGenerationProgress(LogitsProcessor):
    """在每个解码步更新进度，并原样返回候选 logits。"""
    def __init__(self, total, description):
        self.progress = tqdm(
            total=total, desc=description, unit="token-step", dynamic_ncols=True
        )

    def __call__(self, input_ids, scores):
        self.progress.update(1)
        return scores

    def close(self):
        self.progress.close()


def my_generate(prompt: str, do_sample: bool, num_return_sequences: int = 1) -> list[str]:
    """应用 Chat Template 并按确定性或采样策略生成一个或多个回答。"""
    messages = [
        {"role": "system", "content": "你是严谨的计算助手，只依据给定问题作答。"},
        {"role": "user", "content": prompt},
    ]
    encoded = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    encoded = {name: value.to(DEVICE) for name, value in encoded.items()}
    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": do_sample,
        "num_return_sequences": num_return_sequences,
        "pad_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        # 0.7/0.9 仅为 Self-Consistency 提供候选多样性；模型或任务变化后按质量、分歧与成本重调。
        generation_kwargs.update({"temperature": 0.7, "top_p": 0.9})
    progress = MyGenerationProgress(MAX_NEW_TOKENS, "生成推理回答")
    try:
        generated = model.generate(
            **encoded, **generation_kwargs,
            logits_processor=LogitsProcessorList([progress]),
        )
    finally:
        progress.close()
    prompt_length = encoded["input_ids"].shape[1]
    return tokenizer.batch_decode(generated[:, prompt_length:], skip_special_tokens=True)


baseline_outputs = {name: my_generate(prompt, do_sample=False)[0] for name, prompt in strategies.items()}
baseline_outputs


### 3.3．Self-Consistency 最终答案聚合

Self-Consistency 对同一问题采样多条路径，抽取结构化最终答案后多数投票。它适用于存在多种可行推理路径的任务，但会近似线性增加 Decode 成本；如果候选共享同一个系统性错误，多数票仍可能保留系统性错误。五条候选是奇数，却仍可能在三个以上答案之间出现 `2-2-1` 平票；原理实现按 `Counter` 的首次出现顺序处理平票，用于呈现聚合数据流。生产策略需要定义平票规则、`<UNPARSED>` 的处理方式、最低有效候选数和最低票差；共识不足时转交验证器、工具或人工流程。


In [ ]:
FINAL_ANSWER_PATTERN = re.compile(r"最终答案[：:]\s*([^\n]+)")


def my_extract_final_answer(text: str) -> str:
    """提取最后一个结构化最终答案，解析失败时返回哨兵值。"""
    matches = FINAL_ANSWER_PATTERN.findall(text)
    return matches[-1].strip() if matches else "<UNPARSED>"


# 五条候选用于观察投票与成本；候选数增加会近似线性增加 Token 消耗，多类别平票须另定规则。
def my_self_consistency(prompt: str, candidates: int = 5) -> dict[str, object]:
    """采样多个推理结果并对解析后的最终答案执行多数投票。"""
    outputs = my_generate(prompt, do_sample=True, num_return_sequences=candidates)
    answers = [my_extract_final_answer(output) for output in outputs]
    vote_counts = Counter(answers)
    selected_answer, votes = vote_counts.most_common(1)[0]
    return {
        "selected_answer": selected_answer,
        "votes": votes,
        "vote_counts": dict(vote_counts),
        "outputs": outputs,
    }


self_consistent_result = my_self_consistency(strategies["few_shot_cot"])
{key: value for key, value in self_consistent_result.items() if key != "outputs"}


#### 3.3.1．机制可视化：候选最终答案与多数投票

**学习问题**：五次采样得到的结构化最终答案如何形成票数分布，聚合器选择的答案具有多大的票差与解析覆盖？

**验收不变量**：每根柱只读取 `self_consistent_result["vote_counts"]` 中的最终答案计数，柱高总和必须等于本次实际候选数；高亮项必须等于 `selected_answer`，其票数必须等于 `votes` 且达到当前最大计数。图中不读取、不显示 `outputs` 的中间步骤或原始推理文本。


In [ ]:
import matplotlib.pyplot as plt

self_consistency_vote_counts = {
    str(answer): int(count)
    for answer, count in self_consistent_result["vote_counts"].items()
}
self_consistency_candidate_count = len(self_consistent_result["outputs"])
if sum(self_consistency_vote_counts.values()) != self_consistency_candidate_count:
    raise RuntimeError("最终答案票数之和与实际采样候选数不一致")
selected_answer_for_plot = str(self_consistent_result["selected_answer"])
selected_votes_for_plot = int(self_consistent_result["votes"])
if self_consistency_vote_counts.get(selected_answer_for_plot) != selected_votes_for_plot:
    raise RuntimeError("聚合答案的保存票数与 vote_counts 不一致")
if selected_votes_for_plot != max(self_consistency_vote_counts.values()):
    raise RuntimeError("聚合答案没有取得当前最高票数")

vote_rows = sorted(
    self_consistency_vote_counts.items(), key=lambda item: (-item[1], item[0])
)
answer_labels = [answer for answer, _ in vote_rows]
answer_votes = [votes for _, votes in vote_rows]
bar_colors = [
    "#2563eb" if answer == selected_answer_for_plot else "#94a3b8"
    for answer in answer_labels
]
figure, axis = plt.subplots(figsize=(9, max(3.5, 0.75 * len(vote_rows) + 2)), constrained_layout=True)
bars = axis.barh(range(len(vote_rows)), answer_votes, color=bar_colors, height=0.62)
for bar, votes in zip(bars, answer_votes):
    axis.text(votes + 0.05, bar.get_y() + bar.get_height() / 2, str(votes), va="center", fontweight="bold")
axis.set(
    yticks=range(len(vote_rows)), yticklabels=answer_labels,
    xlabel="候选票数", ylabel="解析后的最终答案",
    title=f"Self-Consistency 最终答案投票（候选数={self_consistency_candidate_count}）",
    xlim=(0, self_consistency_candidate_count + 0.6),
)
axis.set_xticks(range(self_consistency_candidate_count + 1))
axis.invert_yaxis()
axis.grid(axis="x", alpha=0.22)
plt.show()


**应观察结论**：蓝色柱对应聚合器实际选择的最终答案，柱高显示其共识强度；其他柱表示竞争答案，`<UNPARSED>` 若出现则单独计入解析失败。最高票与次高票的差距越小，当前候选集合的聚合稳定性越弱，应进入预先规定的平票、验证器或人工升级路径。

**不可误读边界**：多数票只说明采样候选之间的一致性，不证明答案正确，也不表示候选相互独立。相同模型、Prompt 和解码设置可能产生相关的系统性错误；五个候选与单一 Seed 仅用于观察聚合机制，不能据此估计生产准确率。图中只呈现解析后的最终答案，原始思维链不属于可视化或审计证据。


## 4．证据验证

### 4.1．提示策略的统一评测

评测集至少按直接问答、组合运算、约束满足、领域知识和不可回答问题分层。对每个策略记录：

- 最终答案准确率和解析失败率；
- 可验证步骤的一致性，而不是文字长度；
- 输入/输出 Token、TTFT、TPOT 和总成本；
- 对措辞、示例顺序、无关示例和恶意输入的稳定性；
- 不确定时拒答或调用工具的正确率。

代码只用 1 道期望答案为 `150` 的题验证指标 Schema，不能比较四种策略的优劣。正式评测应在运行前按任务层、难度、语言和长度确定样本量；Greedy 策略使用成对题目比较，采样策略还要在预先声明的多个 Seed 上重复，报告每层分子/分母、均值、离散程度或置信区间。Exact Match 需固定大小写、空白、单位和数值格式的归一化规则，解析失败单独报告；报告覆盖全部冻结样本和预先声明的 Seed，并分别呈现解析失败与分层结果。


In [ ]:
EXPECTED_ANSWER = "150"


def my_prompt_metrics(prompt: str, output: str, expected_answer: str) -> dict[str, object]:
    """统计 Prompt/输出 Token，并评估答案可解析性与精确匹配。"""
    extracted = my_extract_final_answer(output)
    return {
        "input_tokens": len(tokenizer.encode(prompt, add_special_tokens=False)),
        "output_tokens": len(tokenizer.encode(output, add_special_tokens=False)),
        "parsed": extracted != "<UNPARSED>",
        "exact_match": extracted == expected_answer,
        "extracted_answer": extracted,
    }


strategy_metrics = {
    name: my_prompt_metrics(strategies[name], output, EXPECTED_ANSWER)
    for name, output in baseline_outputs.items()
}
strategy_metrics


#### 4.1.1．机制可视化：提示策略质量—Token 成本

**学习问题**：四种 Greedy 提示策略与五候选 Self-Consistency 在本次真实生成记录上分别消费多少 Token，额外采样成本是否换来了最终答案 Exact Match 的变化？

**验收不变量**：四个基线点的横坐标严格等于 `strategy_metrics` 中 `input_tokens + output_tokens`，纵坐标严格等于其 `exact_match`。Self-Consistency 横坐标按实际候选数重复计算同一 Prompt 输入，并加总 `self_consistent_result["outputs"]` 的实际输出 Token；纵坐标只比较聚合后的 `selected_answer` 与 `EXPECTED_ANSWER`。原始推理文本只参与 Token 长度计数，不被展示，也不作为质量评分对象。


In [ ]:
strategy_quality_cost_rows = []
for strategy_name, metrics in strategy_metrics.items():
    input_tokens = int(metrics["input_tokens"])
    output_tokens = int(metrics["output_tokens"])
    strategy_quality_cost_rows.append({
        "strategy": strategy_name,
        "total_tokens": input_tokens + output_tokens,
        "exact_match": int(bool(metrics["exact_match"])),
        "kind": "greedy",
    })

self_consistency_outputs = self_consistent_result["outputs"]
self_consistency_input_tokens_per_candidate = len(
    tokenizer.encode(strategies["few_shot_cot"], add_special_tokens=False)
)
self_consistency_total_input_tokens = (
    self_consistency_input_tokens_per_candidate * len(self_consistency_outputs)
)
self_consistency_total_output_tokens = sum(
    len(tokenizer.encode(output, add_special_tokens=False))
    for output in self_consistency_outputs
)
self_consistency_strategy_name = f"self_consistency_{len(self_consistency_outputs)}"
strategy_quality_cost_rows.append({
    "strategy": self_consistency_strategy_name,
    "total_tokens": self_consistency_total_input_tokens + self_consistency_total_output_tokens,
    "exact_match": int(str(self_consistent_result["selected_answer"]) == EXPECTED_ANSWER),
    "kind": "sample_and_vote",
})
if any(row["total_tokens"] <= 0 or row["exact_match"] not in {0, 1} for row in strategy_quality_cost_rows):
    raise RuntimeError("策略质量—成本记录不满足正 Token 数与二元 Exact Match 契约")

figure, axis = plt.subplots(figsize=(11, 5.8), constrained_layout=True)
annotation_offsets = [
    (8, 8) if row["kind"] == "sample_and_vote" else (6, 8 if index % 2 == 0 else -16)
    for index, row in enumerate(strategy_quality_cost_rows)
]
for row, text_offset in zip(strategy_quality_cost_rows, annotation_offsets):
    is_self_consistency = row["kind"] == "sample_and_vote"
    color = "#059669" if row["exact_match"] else "#dc2626"
    axis.scatter(
        row["total_tokens"], row["exact_match"],
        s=130 if is_self_consistency else 80,
        marker="D" if is_self_consistency else "o",
        color=color, edgecolor="white", linewidth=0.8, zorder=3,
    )
    axis.annotate(
        row["strategy"], (row["total_tokens"], row["exact_match"]),
        xytext=text_offset, textcoords="offset points", fontsize=9,
    )
greedy_cot_row = next(row for row in strategy_quality_cost_rows if row["strategy"] == "few_shot_cot")
self_consistency_row = next(row for row in strategy_quality_cost_rows if row["strategy"] == self_consistency_strategy_name)
axis.plot(
    [greedy_cot_row["total_tokens"], self_consistency_row["total_tokens"]],
    [greedy_cot_row["exact_match"], self_consistency_row["exact_match"]],
    color="#64748b", linestyle=":", linewidth=1.2, zorder=1,
)
axis.set(
    xlabel="总 Token（本章统一计量口径）", ylabel="最终答案 Exact Match",
    yticks=[0, 1], yticklabels=["0：未命中", "1：命中"],
    ylim=(-0.18, 1.18), title="提示策略的实际 Token 消费与单题质量结果",
)
axis.grid(alpha=0.22)
plt.show()


**应观察结论**：Self-Consistency 菱形点必然位于 `few_shot_cot` 右侧，因为相同输入被实际采样五次并产生五份输出；其纵向位置仅由多数票最终答案是否等于 `150` 决定。其他点呈现示例与步骤要求对本次输入、输出 Token 的实际影响，绿色与红色分别表示本题 Exact Match 成功与失败。

**不可误读边界**：单道算术题的二元结果不能比较策略总体质量，也不能证明某个 Prompt 策略占优。横轴沿用本章 `tokenizer.encode` 的教学计量口径，未包含 Chat Template 特殊 Token、批处理、KV Cache、墙钟延迟或供应商计费；Self-Consistency 的额外 Token 也不能直接换算为固定倍数的美元成本。原始 CoT 没有进入图形或质量判定。


## 5．迁移到生产库

### 5.1．原理对象与 Transformers 对象映射

| 原理对象 | 生产对象 | 需要固定的契约 |
|---|---|---|
| Example Bank 与示例选择器 | 版本化示例资产、检索器或配置服务 | 来源、任务标签、权限、排序规则和 Token 预算 |
| Prompt Builder | `tokenizer.apply_chat_template` | System、Example、Query 的角色与分隔符 |
| 生成策略 | `GenerationConfig` 与 `model.generate` | `do_sample`、Temperature、Top-p、长度、Seed 与 Stop/EOS |
| 最终答案解析器 | 结构化输出 Schema 或版本化 Parser | 格式、归一化、解析失败和向后兼容性 |
| Self-Consistency 聚合器 | 批量候选生成、Verifier 与投票策略 | 候选数、平票、最低共识、超时与费用 |
| 评测记录 | 评测平台与运行 Manifest | 模型、模板、示例、解析器、Seed、Token、延迟和成本 |

### 5.2．迁移验证

迁移后的 Chat Template 与生成接口使用同一问题、示例和输出格式。输入 Token、消息角色、最终答案解析、Greedy 基线和采样候选需要与原理实现保持语义一致，并记录库默认值差异。


## 6．生产边界

### 6.1．资产、预算与安全边界

1. Prompt、Example Bank、Chat Template、模型 ID 和解析器必须共同版本化。
2. 示例选择先做权限和安全过滤，再做相关性排序。
3. ICL 示例不能覆盖系统政策，也不能携带其他租户数据。
4. CoT 文本可能合理但结论错误；数学、代码、检索和业务规则优先交给可验证工具。
5. 对外返回简洁依据、引用和最终答案；内部只记录必要的结构化决策与工具 Trace。
6. Self-Consistency 必须设置候选数、Token、超时和费用上限；候选数由正确率—成本曲线和投票稳定度决定，输出预算由截断率决定，总超时与费用由请求 SLO 和任务价值决定，达到边际收益后停止增加。
7. Prompt Injection、越权工具调用和敏感信息处理遵循 `70` 的安全边界与 `90` 的应用 Runtime 控制。

ICL 与 CoT 规定模型在运行时的任务组织方式。接入 `90` 的完整应用链路时，Prompt 策略应作为版本化组件并入既有检索、工具、状态、预算、Skill 与轨迹评测，无需另建执行链。

### 6.2．参考资料

- [Hugging Face Transformers：Chat Templates](https://huggingface.co/docs/transformers/main/en/chat_templating)
- [Hugging Face Transformers：Generation](https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
- [Chain-of-Thought Prompting Elicits Reasoning in Large Language Models](https://arxiv.org/abs/2201.11903)
- [Self-Consistency Improves Chain of Thought Reasoning in Language Models](https://arxiv.org/abs/2203.11171)
